# 11 · DistilBERT multilingüe: romper la barrera del idioma — Grupo 17 · vía B
**GPU obligatoria.** Reanudable: cada experimento guarda su JSON y se omite si ya existe.

Modelo: `distilbert-base-multilingual-cased`. Datos: `emails_v1` (inglés nativo) + `emails_es_v2` (español nativo SpaPhish/SpearPhishMX
+ espejo traducido de las 3 clases). **Regla de oro:** el español NATIVO solo se usa para EVALUAR, nunca para entrenar; el traducido solo ENTRENA.

| Exp. | Entrena con | Mide |
|---|---|---|
| E0 | Inglés nativo | Transferencia cross-lingual pura al español nativo |
| E1 | Inglés nativo + español traducido (espejo) | Aporte del espejo traducido |
| E2 | E1 + español nativo (solo su split de train) | Aporte del dato nativo; su test sigue sin verse |
| E2-LOSO | E2 sin una fuente nativa | ¿Generaliza entre SpaPhish y SpearPhishMX? |

Cada experimento se evalúa en: test inglés (no retroceso), **test español NATIVO** (la cifra que importa) y por fuente nativa (SpaPhish / SpearPhishMX).
**Nota declarada:** no hay spam nativo en español; el test nativo mide sobre todo phishing vs legítimo.

In [1]:
# Celda 1: Entorno
import os, sys, json, time, random
from pathlib import Path
import numpy as np, pandas as pd
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
elif ENV == "kaggle":
    h = sorted(Path("/kaggle/input").rglob("emails_v1.parquet")); assert h; BASE = h[0].parent.parent.parent  # ajustar si hace falta
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
PROC, RES, MODELS = BASE / "data/processed", BASE / "results", BASE / "models"
for p in (RES, MODELS): p.mkdir(parents=True, exist_ok=True)
CFG = {"modelo": "distilbert-base-multilingual-cased", "max_len": 256, "batch": 32, "epocas": 3, "lr": 5e-5,
       "warmup": 0.1, "semilla": 17, "paciencia": 1, "experimentos": ["E0_solo_ingles", "E1_ingles_mas_es_traducido", "E2_con_es_nativo",
                        "E2_loso_SpaPhish", "E2_loso_SpearPhishMX"]}
MARCADORES = ["__url__", "__ip__", "__email__", "__anio__", "__dominio_propio__"]
CLASES = ["legitimo", "spam", "phishing"]; PH = 2
print(ENV, BASE)

Mounted at /content/drive
colab /content/drive/MyDrive/PI_G17


In [2]:
# Celda 2: Dependencias, GPU y datos
%pip install -q "transformers>=4.40" accelerate
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
def fijar(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s); os.environ["PYTHONHASHSEED"] = str(s)
fijar(CFG["semilla"])
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEV.type == "cuda" or ENV == "local", "Activa GPU"
GPU = torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu"
COLS = ["id", "grupo", "clase", "y", "split", "texto_modelo"]
EN = pd.read_parquet(PROC / "emails_v1.parquet", columns=COLS).assign(idioma="en", origen="nativo")
ES = pd.read_parquet(PROC / "emails_es_v2.parquet", columns=COLS)
ES["idioma"] = "es"
ES["origen"] = np.where(ES["grupo"].astype(str).str.startswith("traducido_"), "traducido", "nativo")
print(GPU, "| inglés:", len(EN), "| español:", len(ES),
      "| es nativo por clase:", ES[(ES.origen=="nativo")].groupby("clase").size().to_dict())

Tesla T4 | inglés: 54684 | español: 55130 | es nativo por clase: {'legitimo': 1691, 'phishing': 2577}


In [3]:
# Celda 3: Métricas (idénticas a 05/06)
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix
def umbral_recall(y, p, obj=0.95):
    s = np.sort(p[y == PH])[::-1]; return float(s[min(int(np.ceil(obj*len(s)))-1, len(s)-1)]) if len(s) else .5
def predecir(P, u): pred = P[:, :2].argmax(1); pred[P[:, PH] >= u] = PH; return pred
def met(y, pred):
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0,1,2], zero_division=0); no = y != PH
    return {"f1_macro": round(f1_score(y, pred, average="macro", labels=[0,1,2], zero_division=0), 4),
            "precision": dict(zip(CLASES, np.round(p,4).tolist())), "recall": dict(zip(CLASES, np.round(r,4).tolist())),
            "fpr_phishing": round(float((pred[no]==PH).mean()),4) if no.any() else None, "n": int(len(y))}

In [4]:
# Celda 4: Entrenamiento
TOK = AutoTokenizer.from_pretrained(CFG["modelo"]); TOK.add_tokens(MARCADORES)
COL = DataCollatorWithPadding(TOK)
def cod(df):
    e = TOK(df["texto_modelo"].tolist(), truncation=True, max_length=CFG["max_len"])
    return [{"input_ids": a, "attention_mask": m, "labels": int(y)} for a, m, y in zip(e["input_ids"], e["attention_mask"], df["y"])]
def nuevo():
    m = AutoModelForSequenceClassification.from_pretrained(CFG["modelo"], num_labels=3); m.resize_token_embeddings(len(TOK)); return m.to(DEV)
@torch.no_grad()
def proba(m, df, bs=128):
    if not len(df): return np.zeros((0,3))
    m.eval(); out = []
    for b in DataLoader(cod(df), batch_size=bs, collate_fn=COL):
        b = {k: v.to(DEV) for k, v in b.items() if k != "labels"}
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=DEV.type=="cuda"):
            out.append(torch.softmax(m(**b).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)
def entrenar(df_tr, df_va):
    fijar(CFG["semilla"]); m = nuevo(); y = df_tr["y"].values
    w = torch.tensor(len(y)/(3*np.bincount(y, minlength=3)), dtype=torch.float, device=DEV)
    dl = DataLoader(cod(df_tr), batch_size=CFG["batch"], shuffle=True, collate_fn=COL, generator=torch.Generator().manual_seed(CFG["semilla"]))
    opt = torch.optim.AdamW(m.parameters(), lr=CFG["lr"], weight_decay=0.01)
    tot = len(dl)*CFG["epocas"]; sch = get_linear_schedule_with_warmup(opt, int(CFG["warmup"]*tot), tot)
    sc = torch.amp.GradScaler("cuda", enabled=DEV.type=="cuda"); loss_fn = torch.nn.CrossEntropyLoss(weight=w)
    mejor, estado, sin = -1, None, 0
    for e in range(CFG["epocas"]):
        m.train(); t0 = time.time()
        for b in dl:
            b = {k: v.to(DEV) for k, v in b.items()}; yy = b.pop("labels")
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=DEV.type=="cuda"):
                loss = loss_fn(m(**b).logits.float(), yy)
            opt.zero_grad(set_to_none=True); sc.scale(loss).backward(); sc.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); sc.step(opt); sc.update(); sch.step()
        f1v = f1_score(df_va["y"], proba(m, df_va).argmax(1), average="macro")
        print(f"  época {e+1}: f1_val {f1v:.4f} ({time.time()-t0:.0f}s)")
        if f1v > mejor: mejor, sin, estado = f1v, 0, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
        else:
            sin += 1
            if sin > CFG["paciencia"]: break
    m.load_state_dict(estado); return m

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [5]:
# Celda 5: Conjuntos de cada experimento
def conjuntos(exp):
    en_tr, en_va = EN[EN.split=="train"], EN[EN.split=="val"]
    tr_tr = ES[(ES.origen=="traducido") & (ES.split=="train")]
    tr_va = ES[(ES.origen=="traducido") & (ES.split=="val")]
    nat = ES[ES.origen=="nativo"]
    if exp == "E0_solo_ingles":
        return en_tr, en_va
    if exp == "E1_ingles_mas_es_traducido":                 # nunca el nativo
        return pd.concat([en_tr, tr_tr]), pd.concat([en_va, tr_va])
    if exp == "E2_con_es_nativo":                           # + español NATIVO de train (su test queda intacto)
        return (pd.concat([en_tr, tr_tr, nat[nat.split=="train"]]),
                pd.concat([en_va, tr_va, nat[nat.split=="val"]]))
    g = exp.replace("E2_loso_", "")                         # una fuente nativa FUERA por completo
    otra = nat[nat.grupo != g]
    return (pd.concat([en_tr, tr_tr, otra[otra.split=="train"]]),
            pd.concat([en_va, tr_va, otra[otra.split=="val"]]))
# Conjuntos de EVALUACIÓN (fijos para ambos experimentos): inglés test + ESPAÑOL NATIVO test
EVAL = {"ingles_test": EN[EN.split=="test"],
        "es_nativo_test": ES[(ES.origen=="nativo") & (ES.split=="test")],
        "es_nativo_SpaPhish": ES[(ES.origen=="nativo") & (ES.grupo=="SpaPhish")],
        "es_nativo_SpearPhishMX": ES[(ES.origen=="nativo") & (ES.grupo=="SpearPhishMX")]}
{k: {"n": len(v), "clases": v.groupby("clase").size().to_dict()} for k, v in EVAL.items()}

{'ingles_test': {'n': 8203,
  'clases': {'legitimo': 3207, 'phishing': 1488, 'spam': 3508}},
 'es_nativo_test': {'n': 641, 'clases': {'legitimo': 254, 'phishing': 387}},
 'es_nativo_SpaPhish': {'n': 1343,
  'clases': {'legitimo': 632, 'phishing': 711}},
 'es_nativo_SpearPhishMX': {'n': 2925,
  'clases': {'legitimo': 1059, 'phishing': 1866}}}

In [6]:
# Celda 6: Correr experimentos (reanudable)
def correr(exp):
    dest = RES / f"multilingue_{exp}.json"
    if dest.exists(): print(exp, "ya existe"); return json.loads(dest.read_text())
    print("==", exp); t0 = time.time(); tr, va = conjuntos(exp)
    m = entrenar(tr, va)
    Pva = proba(m, va); u = umbral_recall(va["y"].values, Pva[:, PH])
    R = {"experimento": exp, "n_train": len(tr), "umbral_val": round(u,4), "gpu": GPU, "evaluaciones": {}}
    evals = dict(EVAL)
    if exp.startswith("E2_loso_"):
        g = exp.replace("E2_loso_", "")
        evals = {"ingles_test": EVAL["ingles_test"], f"fuera_{g}": ES[(ES.origen=="nativo") & (ES.grupo==g)]}
    for nombre, df in evals.items():
        if not len(df): R["evaluaciones"][nombre] = {"n": 0}; continue
        P = proba(m, df); y = df["y"].values
        R["evaluaciones"][nombre] = {"umbral_recall95": met(y, predecir(P, u)), "argmax": met(y, P.argmax(1))}
        # phishing vs legítimo (relevante en español, sin spam nativo)
        pl = df["clase"].isin(["legitimo","phishing"]).values
        if pl.any():
            yb = (y[pl]==PH).astype(int);
            from sklearn.metrics import roc_auc_score
            R["evaluaciones"][nombre]["auc_phishing_vs_legitimo"] = round(float(roc_auc_score(yb, P[pl, PH])), 4) if len(set(yb))>1 else None
    if exp == "E2_con_es_nativo":
        m.save_pretrained(MODELS / "distilbert_multi_v1"); TOK.save_pretrained(MODELS / "distilbert_multi_v1")
        # probabilidades del español nativo (val+test) para la fusión en español (12)
        esd = ES[(ES.origen=="nativo") & (ES.split.isin(["val","test"]))]
        Pe = proba(m, esd)
        pd.DataFrame({"id": esd["id"].values, "split": esd["split"].values,
                      "p_legitimo": Pe[:,0], "p_spam": Pe[:,1], "p_phishing": Pe[:,2]}).to_parquet(PROC / "text_scores_multi_es_v1.parquet", index=False)
    R["segundos"] = round(time.time()-t0); dest.write_text(json.dumps(R, indent=1, ensure_ascii=False, default=str))
    del m; torch.cuda.empty_cache() if DEV.type=="cuda" else None
    clave = next(k for k in R["evaluaciones"] if k != "ingles_test")      # es_nativo_test o fuera_<fuente>
    print(exp, clave, "recall:", R["evaluaciones"][clave].get("umbral_recall95", {}).get("recall"))
    return R
RES_EXP = {e: correr(e) for e in CFG["experimentos"]}

E0_solo_ingles ya existe
E1_ingles_mas_es_traducido ya existe
E2_con_es_nativo ya existe
E2_loso_SpaPhish ya existe
== E2_loso_SpearPhishMX


model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/voca

  época 1: f1_val 0.9739 (532s)
  época 2: f1_val 0.9844 (528s)
  época 3: f1_val 0.9842 (530s)
E2_loso_SpearPhishMX fuera_SpearPhishMX recall: {'legitimo': 0.6242, 'spam': 0.0, 'phishing': 0.0761}


In [7]:
# Celda 7: Comparación y RESUMEN
def fila(R, clave):
    e = R["evaluaciones"].get(clave, {})
    return {"recall_phishing": e.get("umbral_recall95", {}).get("recall", {}).get("phishing"),
            "f1_macro": e.get("umbral_recall95", {}).get("f1_macro"),
            "auc_ph_vs_leg": e.get("auc_phishing_vs_legitimo")}
PRINC = ["E0_solo_ingles", "E1_ingles_mas_es_traducido", "E2_con_es_nativo"]
comp = {clave: {e: fila(RES_EXP[e], clave) for e in PRINC}
        for clave in ("ingles_test", "es_nativo_test", "es_nativo_SpaPhish", "es_nativo_SpearPhishMX")}
loso = {g: fila(RES_EXP[f"E2_loso_{g}"], f"fuera_{g}") for g in ("SpaPhish", "SpearPhishMX")}
RESUMEN = {"entorno": ENV, "gpu": GPU, "modelo": CFG["modelo"],
           "n_train": {e: RES_EXP[e]["n_train"] for e in CFG["experimentos"]},
           "comparacion_E0_E1_E2": comp, "E2_fuente_es_fuera": loso,
           "detalle_es_nativo_test": {e: RES_EXP[e]["evaluaciones"]["es_nativo_test"] for e in PRINC}}
(RES / "multilingue_resumen.json").write_text(json.dumps(RESUMEN, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (11) =====")
print(json.dumps(RESUMEN, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (11) =====
{"entorno":"colab","gpu":"Tesla T4","modelo":"distilbert-base-multilingual-cased","n_train":{"E0_solo_ingles":38278,"E1_ingles_mas_es_traducido":73881,"E2_con_es_nativo":76868,"E2_loso_SpaPhish":75928,"E2_loso_SpearPhishMX":74821},"comparacion_E0_E1_E2":{"ingles_test":{"E0_solo_ingles":{"recall_phishing":0.9536,"f1_macro":0.9831,"auc_ph_vs_leg":0.9996},"E1_ingles_mas_es_traducido":{"recall_phishing":0.9778,"f1_macro":0.9913,"auc_ph_vs_leg":0.9999},"E2_con_es_nativo":{"recall_phishing":0.9724,"f1_macro":0.9889,"auc_ph_vs_leg":0.9999}},"es_nativo_test":{"E0_solo_ingles":{"recall_phishing":0.6331,"f1_macro":0.2776,"auc_ph_vs_leg":0.5665},"E1_ingles_mas_es_traducido":{"recall_phishing":0.6279,"f1_macro":0.3576,"auc_ph_vs_leg":0.5996},"E2_con_es_nativo":{"recall_phishing":0.9354,"f1_macro":0.6296,"auc_ph_vs_leg":0.9844}},"es_nativo_SpaPhish":{"E0_solo_ingles":{"recall_phishing":0.7637,"f1_macro":0.3748,"auc_ph_vs_leg":0.7722},"E1_ingles_mas_es_traducido":{"recall_ph